In [100]:
with open('input.txt', 'r', encoding='utf-8') as f:
    data = f.read()

In [101]:
#normalize
data = data.lower()

In [102]:
len(data)

19770

In [103]:
#pre-tokenization
train, test = data[:int(0.8*len(data))], data[int(0.8*len(data)):]

In [104]:
len(train), len(test)

(15816, 3954)

In [105]:
vocab = list(set(list(train)))

In [106]:
## creating training corpus

initial_corpus = []
training_words = train.split()
initial_corpus = [list(word) + ['</w>'] for word in training_words]

In [122]:
training_words

In [107]:
len(initial_corpus), len(training_words)

(2004, 2004)

In [108]:
initial_corpus[:5]

[['f', 'o', 'u', 'n', 'd', 'a', 't', 'i', 'o', 'n', 's', '</w>'],
 ['o', 'f', '</w>'],
 ['w', 'o', 'r', 'l', 'd', '</w>'],
 ['p', 'o', 'l', 'i', 't', 'i', 'c', 's', '</w>'],
 ['w', 'o', 'r', 'l', 'd', '</w>']]

In [109]:
### Helper Functions

def calculate_frequency_pairs(corpus):
    freq_pairs = {}
    for word in corpus:
        for i in range(len(word)-1):
            freq_pairs[(word[i],word[i+1])] = freq_pairs.get((word[i],word[i+1]), 0) + 1
    return freq_pairs

In [111]:
calculate_frequency_pairs(initial_corpus[:1])

{('f', 'o'): 1,
 ('o', 'u'): 1,
 ('u', 'n'): 1,
 ('n', 'd'): 1,
 ('d', 'a'): 1,
 ('a', 't'): 1,
 ('t', 'i'): 1,
 ('i', 'o'): 1,
 ('o', 'n'): 1,
 ('n', 's'): 1,
 ('s', '</w>'): 1}

In [112]:
## Merge and Update corpus with merged token

def merge_and_update_corpus(corpus, merged_pair, freq_pairs):
    merged_token = merged_pair[0] + merged_pair[1]
    new_corpus = []
    for word in corpus:
        new_word = []
        i = 0
        while i < (len(word)):
            if (i < len(word)-1) and (word[i],word[i+1]) == merged_pair:
                new_word.append(merged_token)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_corpus.append(new_word)
    return new_corpus, merged_token

In [113]:
training_iter = 500
vocab_size = 1000
i = 0
merge_rules = []
corpus = initial_corpus
while i < training_iter and len(vocab) < vocab_size:
    freq_pairs = calculate_frequency_pairs(corpus)
    curr_merge_pair = max(freq_pairs, key=freq_pairs.get)
    corpus, merged_token = merge_and_update_corpus(corpus, curr_merge_pair, freq_pairs)
    print(len(corpus))
    merge_rules.append(curr_merge_pair)
    print(f"Current merge rule: {merged_token} from merge_pair: {curr_merge_pair}")
    vocab.append(merged_token)
    i += 1


2004
Current merge rule: in from merge_pair: ('i', 'n')
2004
Current merge rule: on from merge_pair: ('o', 'n')
2004
Current merge rule: er from merge_pair: ('e', 'r')
2004
Current merge rule: s</w> from merge_pair: ('s', '</w>')
2004
Current merge rule: ti from merge_pair: ('t', 'i')
2004
Current merge rule: e</w> from merge_pair: ('e', '</w>')
2004
Current merge rule: th from merge_pair: ('t', 'h')
2004
Current merge rule: al from merge_pair: ('a', 'l')
2004
Current merge rule: ,</w> from merge_pair: (',', '</w>')
2004
Current merge rule: an from merge_pair: ('a', 'n')
2004
Current merge rule: en from merge_pair: ('e', 'n')
2004
Current merge rule: tion from merge_pair: ('ti', 'on')
2004
Current merge rule: or from merge_pair: ('o', 'r')
2004
Current merge rule: y</w> from merge_pair: ('y', '</w>')
2004
Current merge rule: d</w> from merge_pair: ('d', '</w>')
2004
Current merge rule: al</w> from merge_pair: ('al', '</w>')
2004
Current merge rule: st from merge_pair: ('s', 't')
2004
C

In [114]:
len(vocab),  len(merge_rules)

(533, 500)

In [115]:
merge_rules[0]

('i', 'n')

### Inference

In [116]:
##
def pre_tokenize(sentence):
    words = sentence.split()
    input_sent = []
    for word in words:
        input_sent.extend(list(word))
        input_sent.append('</w>')

    return input_sent

In [117]:
sentence = "All the lights we cannot see"

In [118]:
input_words = pre_tokenize(sentence)
input_words

['A',
 'l',
 'l',
 '</w>',
 't',
 'h',
 'e',
 '</w>',
 'l',
 'i',
 'g',
 'h',
 't',
 's',
 '</w>',
 'w',
 'e',
 '</w>',
 'c',
 'a',
 'n',
 'n',
 'o',
 't',
 '</w>',
 's',
 'e',
 'e',
 '</w>']

In [119]:
def tokenize_word(word, merge_rules):

    tokens = list(word.lower()) + ['</w>']

    for pair in merge_rules:

        merged = pair[0] + pair[1]

        i = 0
        new_tokens = []

        while i < len(tokens):

            if (
                i < len(tokens)-1 and
                (tokens[i], tokens[i+1]) == pair
            ):
                new_tokens.append(merged)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1

        tokens = new_tokens

    return tokens

In [120]:
sentence = "All the lights we cannot see"

all_tokens = []
for word in sentence.split():
    all_tokens.extend(tokenize_word(word, merge_rules))

In [121]:
print(all_tokens)

['all</w>', 'the</w>', 'li', 'g', 'h', 'ts</w>', 'w', 'e</w>', 'c', 'an', 'not</w>', 'se', 'e</w>']
